# Check VAE: load checkpoint, sample random latent, reconstruct examples and show differences

This notebook loads a trained VAE checkpoint, reconstructs validation samples, samples a random latent, decodes it, and visualizes input / reconstruction / absolute difference / random sample for each channel.


In [2]:
# Imports and device
import torch
import numpy as np
import matplotlib.pyplot as plt
from autoencoderldm3d import AutoencoderKL, ddconfig, lossconfig, ContinuousVAELoss
from config_vae import config as _config
from dataset import GridDataModule
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# simple helper: map from [-1,1] -> [0,1] for display
unnormalize = lambda x: (x + 1.0) / 2.0


Using device: cuda


In [4]:
# Load model from checkpoint
cfg = _config()

# architecture config same as training
ae_ddconfig = ddconfig(
    double_z=True,
    z_channels=4,
    resolution=32,
    in_channels=3,
    out_ch=3,
    ch=32,
    ch_mult=[1,2,4,4],
    num_res_blocks=2,
    dropout=0.0,
    has_mid_attn=True,
)

ae_lossconfig = lossconfig(target=ContinuousVAELoss, kl_weight=1e-6)

model = AutoencoderKL(ddconfig=ae_ddconfig, lossconfig=ae_lossconfig, embed_dim=4)

# path to checkpoint (adjust if you want a different ckpt)
ckpt_path = os.path.join('checkpoints_ae','vae_lim_1.0','ae_epoch=096_val_loss=0.003791.ckpt')
print('Looking for checkpoint at', ckpt_path)

# load weights
model.init_from_ckpt(ckpt_path)
model.to(device)
model.eval()
print('Model restored.')


making attention of type 'vanilla' with 128 in_channels
Working with z of shape (1, 4, 4, 4, 4) = 256 dimensions.
making attention of type 'vanilla' with 128 in_channels
Looking for checkpoint at checkpoints_ae/vae_lim_1.0/ae_epoch=096_val_loss=0.003791.ckpt
Restored from checkpoints_ae/vae_lim_1.0/ae_epoch=096_val_loss=0.003791.ckpt
Model restored.


## Data preparation

In [5]:
# Prepare datamodule and fetch a batch
from torch.utils.data import DataLoader

cfg["train"] = False  # ensure we are in eval mode
cfg["train_dataset"] = "../data/train"
cfg["batch_size"] = 4

dm = GridDataModule(cfg)
# ensure datamodule datasets are prepared
try:
    dm.setup(stage=None)
except Exception as e:
    print('Warning during datamodule setup:', e)

val_loader = dm.test_dataloader()
# grab a small batch
batch = next(iter(val_loader))
inputs, props = batch
print('Batch shapes:', inputs.shape, props.shape)
inputs = inputs.float().to(device)


Batch shapes: torch.Size([4, 3, 32, 32, 32]) torch.Size([4, 3])


In [6]:
cfg

{'exp_name': 'vae',
 'train': False,
 'seed': 42,
 'batch_size': 4,
 'max_epochs': 200,
 'lr': 0.001,
 'accelerator': 'gpu',
 'devices': 1,
 'n_gpu': 1,
 'precision': 32,
 'save_dir': 'checkpoints_ae',
 'dim': 32,
 'dim_mults': [1, 2, 4],
 'channels': 3,
 'model_dir': 'models/',
 'train_dataset': '../data/train',
 'test_dataset': '../data/test/',
 'num_workers': 8,
 'augmentation': True,
 'test_only_100': False,
 'log_dir': '../logs_ae/',
 'property_file': '../data/properties.pickle',
 'c_model_dir': 'models/lattice_regressor.ckpt',
 'c_dim_mults': [1, 2, 4]}

## Reevaluate the model

In [7]:
# Encode, reconstruct and sample random latent; compute MSE
with torch.no_grad():
    recon, posterior = model(inputs)
    # posterior.sample() returns a latent with shape [B, embed_dim, sx, sy, sz]
    z_post = posterior.sample()
    z_rand = torch.randn_like(z_post).to(device)
    dec_rand = model.decode(z_rand)

# compute MSE per sample
mse_per_sample = ((inputs - recon)**2).mean(dim=[1,2,3,4]).cpu().tolist()
print('MSE per sample:', mse_per_sample)


MSE per sample: [0.001551240449771285, 0.0015358811942860484, 0.0031186467967927456, 0.006477816961705685]


In [8]:
from Unet_Reg import Unet_Reg
# Load cell size regressor
cell_model  = Unet_Reg(dim = cfg['dim'],
                       dim_mults = cfg['c_dim_mults'],
                       channels = cfg['channels'],
                    )
cell_model.load_state_dict(torch.load(cfg['c_model_dir']))

/home/ahardiagon/Programs/miniconda3/envs/zeodiff/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<All keys matched successfully>

## Reconstruct some samples from training for visual comparison

In [9]:
from util import write_visit_sample

In [ ]:
def save_recon_vae(inputs, recon,channels=3,grid_size=32,out_dir="../test_vae_output/recons"):
    """ Save input and reconstruction slices for visual inspection """
    os.makedirs(out_dir, exist_ok=True)
    inputs = inputs.cpu().numpy()
    recon = recon.cpu().numpy()
    batch_size = inputs.shape[0]
    
    # compute cell parameters on CPU
    cell_param_list_inputs = cell_model(torch.tensor(inputs, dtype=torch.float32))
    cell_param_list_recon = cell_model(torch.tensor(recon, dtype=torch.float32))
    
    count=0
    for i in range(len(inputs)):
        arr = np.asarray(inputs[i], dtype=np.float32).reshape(channels, grid_size, grid_size, grid_size)
        cell_param = [float(j) * 100 for j in cell_param_list_inputs[i]]
        write_visit_sample(arr, cell=cell_param, stem=f'sample_input_{count}', save_dir=out_dir)
        count+=1

    count=0
    for i in range(len(recon)):
        arr = np.asarray(recon[i], dtype=np.float32).reshape(channels, grid_size, grid_size, grid_size)
        cell_param = [float(j) * 100 for j in cell_param_list_recon[i]]
        write_visit_sample(arr, cell=cell_param, stem=f'sample_recon_{count}', save_dir=out_dir)
        count+=1
    print(f'Reconstructions and inputs saved to {out_dir}')

In [12]:
save_recon_vae(inputs, recon)

NameError: name 'directory' is not defined

## Sample the latent space

In [ ]:
# Sampling helper: generate Visit-format samples from random VAE latents
def vae_large_sample(model, cell_model, num_sample, directory, grid_size=32, channels=3, batch_size=50, device=None):
    os.makedirs(directory, exist_ok=True)
    if device is None:
        device = next(model.parameters()).device

    # ensure cell_model is in eval mode
    cell_model.eval()

    # determine latent shape by encoding a dummy input
    with torch.no_grad():
        dummy = torch.randn(1, channels, grid_size, grid_size, grid_size, device=device)
        _, posterior = model(dummy)
        z_example = posterior.sample()

    count = 1
    num_full = num_sample // batch_size
    left_over = num_sample - num_full * batch_size

    for _ in range(num_full):
        z_rand = torch.randn((batch_size, *z_example.shape[1:]), device=device)
        with torch.no_grad():
            dec_t = model.decode(z_rand)  # torch tensor on `device`
            dec = dec_t.cpu().detach().numpy()

        # Ensure we have a numeric float32 ndarray (avoid object dtype caused by ragged lists)
        if dec.dtype == object:
            print('Warning: decoded array has object dtype — coercing to float32')
            dec = np.stack([np.asarray(x, dtype=np.float32) for x in dec])
        else:
            dec = np.asarray(dec, dtype=np.float32)

        # compute cell parameters on CPU
        cell_param_list = cell_model(torch.tensor(dec, dtype=torch.float32))

        for i in range(len(dec)):
            arr = np.asarray(dec[i], dtype=np.float32).reshape(channels, grid_size, grid_size, grid_size)
            cell_param = [float(j) * 100 for j in cell_param_list[i]]
            write_visit_sample(arr, cell=cell_param, stem=f'sample_{count}', save_dir=directory)
            count += 1

    if left_over > 0:
        z_rand = torch.randn((left_over, *z_example.shape[1:]), device=device)
        with torch.no_grad():
            dec_t = model.decode(z_rand)
            dec = dec_t.cpu().detach().numpy()

        if dec.dtype == object:
            print('Warning: decoded array has object dtype — coercing to float32')
            dec = np.stack([np.asarray(x, dtype=np.float32) for x in dec])
        else:
            dec = np.asarray(dec, dtype=np.float32)

        cell_param_list = cell_model(torch.tensor(dec, dtype=torch.float32))
        for i in range(len(dec)):
            arr = np.asarray(dec[i], dtype=np.float32).reshape(channels, grid_size, grid_size, grid_size)
            cell_param = [float(j) * 100 for j in cell_param_list[i]]
            write_visit_sample(arr, cell=cell_param, stem=f'sample_{count}', save_dir=directory)
            count += 1


# Example invocation (adjust `num_sample` and `directory` as needed)
vae_large_sample(model, cell_model, num_sample=10, directory="../test_vae_output", grid_size=32, channels=3, batch_size=4, device=device)
